In [1]:
import torch
import torch.nn as nn
import FrEIA.framework as Ff
import FrEIA.modules as Fm

In [ ]:
def build_cinn(input_dim, cond_dim, n_blocks=8, hidden_dim=128):
    # Perceptron wielowarstwowy
    def subnet_fc(dims_in, dims_out):
        return nn.Sequential(
            nn.Linear(dims_in, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dims_out)
        )

    # Inicjalizujemy z wymiarem wejściowym
    inn = Ff.SequenceINN(input_dim)

    # Dodajemy bloki sprzęgające z określeniem warunków
    for _ in range(n_blocks):
        inn.append(
            Fm.AllInOneBlock,
            cond=0,                          # Pierwszy warunek (index 0)
            cond_shape=(cond_dim,),          # Kształt tensora warunków (w formie tupli)
            subnet_constructor=subnet_fc,
            permute_soft=False
        )
    return inn

In [ ]:
input_dim = 8575 # Ilość badanych punktów w fluxie (ilość punktów w których badamy natężenie światła)
cond_dim = 3 # Ilość warunków które przewidujemy
model = build_cinn(input_dim, cond_dim)

# Jeśli jest cuda, to ładuje na cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(model)

SequenceINN(
  (module_list): ModuleList(
    (0-7): 8 x AllInOneBlock(
      (softplus): Softplus(beta=0.5, threshold=20.0)
      (subnet): Sequential(
        (0): Linear(in_features=3760, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=7514, bias=True)
      )
    )
  )
)
